# CCT College Dublin - Continuous Assessment 1 (CA1)

**Student Name:** Daniel Adejare

**Student ID:** 2026848

---

| Field | Details |
| :--- | :--- |
| **Programme Title** | BSc Computing and IT (Stage 4, Semester 1) |
| **Cohort Details** | DataVisComms_BScCompIT_Sept23 FT Stage 4 Semester 1 |
| **Module Title** | Data Visualisation and Communication |
| **Lecturer** | Taufique Ahmed |
| **Assignment Title** | CA1 (Individual, 40% Weighting) |
| **Submission Deadline** | 07-11-2026 |

---

### Mandatory Submission Links
* **Screen Capture Video (Google Drive):** [Pending Video Link]

* **GitHub Repository:** https://github.com/Dee-17-C/CA1_Data_Visualisation


---

### Overview
This notebook presents **7 distinct visualisation types** analysing US employment and compensation across Artificial Intelligence (AI), Data Science, and Software Development roles using the May 2023 Bureau of Labor Statistics (BLS) OEWS dataset (U.S. Bureau of Labor Statistics, 2024), alongside multi-year datasets (2021–2023) for the longitudinal Challenge task.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

# Target SOC codes for AI, Data Science & Software Development
TARGET_ROLES = {
    '15-2051': 'Data Scientists',
    '15-1252': 'Software Developers',
    '15-1221': 'Computer & Info Research Scientists (AI)',
    '15-1253': 'Software QA Analysts & Testers',
    '15-1243': 'Database Architects',
    '15-1211': 'Computer Systems Analysts',
    '15-2031': 'Operations Research Analysts',
    '15-1299': 'Computer Occupations, All Other'
}

# Load May 2023 dataset
file_2023 = 'all_data_M_2023.xlsx' if os.path.exists('all_data_M_2023.xlsx') else 'oesm23all/all_data_M_2023.xlsx'
df_raw = pd.read_excel(file_2023, sheet_name=0)
df_raw.columns = df_raw.columns.str.upper()

# Convert BLS suppression symbols ('*', '**', '#') to NaN and cast to float
numeric_cols = [
    'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT',
    'A_MEAN', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90'
]

def clean_bls_numeric(df, cols):
    df_clean = df.copy()
    for col in cols:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(
                df_clean[col].astype(str).str.replace(',', '').replace({'*': np.nan, '**': np.nan, '#': np.nan}),
                errors='coerce'
            )
    return df_clean

df_2023 = clean_bls_numeric(df_raw, numeric_cols)
df_2023['ROLE_NAME'] = df_2023['OCC_CODE'].map(TARGET_ROLES)

# Filter for National Cross-Industry, Cross-Ownership totals (AREA_TYPE == 1, NAICS == '000000', OWN_CODE == 1235)
df_nat_2023 = df_2023[
    (df_2023['AREA_TYPE'] == 1) &
    (df_2023['NAICS'].astype(str).str.startswith('000000')) &
    (df_2023['OWN_CODE'] == 1235) &
    (df_2023['OCC_CODE'].isin(TARGET_ROLES.keys()))
].copy()

df_nat_2023[['OCC_CODE', 'ROLE_NAME', 'TOT_EMP', 'A_MEAN', 'A_MEDIAN']].sort_values(by='TOT_EMP', ascending=False)

## Visualisation 1: Horizontal Bar Chart – National Employment by Role

- **Objective (comparison/ranking):** To compare and rank the size of the US workforce (`TOT_EMP`) across eight AI, Data Science and Software occupations in May 2023.
- **Why this visualisation:** The data pairs one categorical variable (occupation) with one quantitative variable (employment). Bars on a common baseline are the most accurate way to compare magnitudes (Cleveland and McGill, 1984), and horizontal bars keep the long job titles readable.
- **What I learned about AI employment:** Software Developers dominate with 1,656.9K jobs, over three times the next largest role, Computer Systems Analysts (498.8K). Data Scientists (192.7K) are about 5.5 times larger than Computer & Information Research Scientists, the closest SOC category to AI research (35.2K), which is the smallest group. AI-specific research is therefore a very small, specialised part of the tech workforce. The eight roles together employ about 3.2 million people.

In [2]:
v1_data = df_nat_2023.sort_values(by='TOT_EMP', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(v1_data['ROLE_NAME'], v1_data['TOT_EMP'] / 1000, color='#1f77b4', edgecolor='black', height=0.6, alpha=0.88)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 25, bar.get_y() + bar.get_height()/2, f'{width:,.1f}K',
            va='center', ha='left', fontsize=9.5, fontweight='bold', color='#1a1a1a')

ax.set_title('Viz 1: US National Employment by AI, Data Science & Software Role (May 2023)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Total National Employment (Thousands)', fontsize=11)
ax.set_xlim(0, (v1_data['TOT_EMP'].max() / 1000) * 1.18)
ax.grid(axis='x', linestyle='--', alpha=0.6)
plt.figtext(0.01, -0.03, 'Source: U.S. Bureau of Labor Statistics (BLS) OEWS May 2023.', fontsize=9, style='italic')
plt.tight_layout()
plt.show()

## Visualisation 2: Donut Chart – Workforce Composition

- **Objective (composition):** To show how the combined workforce of the eight roles (≈3.2 million) divides between occupations, as a part-to-whole view.
- **Why this visualisation:** Donut charts suit part-to-whole data, but angles are judged less accurately than bar lengths (Cleveland and McGill, 1984). I therefore limited it to five slices by grouping the four smallest roles (Data Scientists, Operations Research Analysts, Database Architects, and Computer & Info Research Scientists) into "Other". Viz 1 covers exact ranking.
- **What I learned about AI employment:** Software Developers make up 51.8% of the workforce, and with Computer Systems Analysts (15.6%) they account for about two-thirds (67%). Adding "Computer Occupations, All Other" (13.7%) takes the top three to 81%. Software QA Analysts are 6.3%. The four specialist roles, including Data Scientists and AI researchers, share just 12.7% between them, so data and AI roles remain a small fraction of tech employment.

In [3]:
v2_sorted = df_nat_2023.sort_values(by='TOT_EMP', ascending=False).copy()
top_4 = v2_sorted.iloc[:4][['ROLE_NAME', 'TOT_EMP']].copy()
other_sum = v2_sorted.iloc[4:]['TOT_EMP'].sum()
donut_df = pd.concat([top_4, pd.DataFrame({'ROLE_NAME': ['Other AI, QA & Data Roles (4 Cats)'], 'TOT_EMP': [other_sum]})], ignore_index=True)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
colors = ['#1b365d', '#2b5c8f', '#4a89c2', '#7bb2dd', '#b6d4ec']
wedges, texts, autotexts = ax.pie(
    donut_df['TOT_EMP'],
    labels=donut_df['ROLE_NAME'],
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    pctdistance=0.77,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2),
    textprops=dict(fontsize=9.5, fontweight='bold')
)
for autotext in autotexts:
    autotext.set_color('white' if autotext.get_text() in ['51.8%', '15.6%'] else 'black')

ax.set_title('Viz 2: Proportional Composition of the US AI, Data & Software Workforce (May 2023)', fontsize=13, fontweight='bold', pad=12)
plt.figtext(0.01, -0.02, 'Source: BLS OEWS May 2023 National Totals.', fontsize=9, style='italic')
plt.tight_layout()
plt.show()

## Visualisation 3: Percentile Range Plot – Salary Distribution

- **Objective (distribution):** To investigate how wages are spread within each role, using the 10th, 25th, 50th (median), 75th and 90th percentiles together with the mean.

- **Why this visualisation:** BLS publishes pre-aggregated percentiles rather than individual salaries, so a standard box plot cannot be drawn. A custom horizontal box-and-whisker plot encodes the same information: the box is the 25th–75th percentile range, the whiskers are the 10th–90th, the white line is the median and the diamond is the mean. Comparing positions along a common axis is accurate (Cleveland and McGill, 1984).

- **What I learned about AI employment:** Computer & Info Research Scientists (AI) have the highest median ($145.1K) and mean ($157.2K), and the widest spread, from about $81K at the 10th percentile to about $233K at the 90th. Operations Research Analysts have the lowest median ($83.6K).

  In every one of the eight roles the mean is above the median, which indicates right-skewed pay where a minority of high earners lift the average. The mean–median gap is largest for AI researchers ($12.1K), Operations Research Analysts ($12.0K) and Data Scientists ($11.0K), and smallest for Database Architects ($2.3K).

In [4]:
v3_data = df_nat_2023.sort_values(by='A_MEDIAN', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11.5, 6))
for idx, row in v3_data.iterrows():
    p10, p25, med, p75, p90, mean = (
        row['A_PCT10']/1000, row['A_PCT25']/1000, row['A_MEDIAN']/1000,
        row['A_PCT75']/1000, row['A_PCT90']/1000, row['A_MEAN']/1000
    )
    if np.isnan(p90):
        p90 = 239.2  # BLS top-coded '#' threshold ($239,200/yr)

    ax.hlines(y=idx, xmin=p10, xmax=p90, color='#555555', linewidth=1.8, zorder=1)
    ax.vlines(x=[p10, p90], ymin=idx-0.15, ymax=idx+0.15, color='#555555', linewidth=1.8, zorder=1)
    ax.barh(y=idx, width=p75-p25, left=p25, height=0.45, color='#4c72b0', alpha=0.78, edgecolor='black', zorder=2)
    ax.vlines(x=med, ymin=idx-0.225, ymax=idx+0.225, color='white', linewidth=2.5, zorder=3)
    ax.scatter(mean, idx, color='#ff7f0e', edgecolor='black', s=65, zorder=4, marker='D')

ax.set_yticks(range(len(v3_data)))
ax.set_yticklabels(v3_data['ROLE_NAME'], fontsize=10, fontweight='bold')
ax.set_xlabel('Annual Wage ($ USD Thousands)', fontsize=11)
ax.set_title('Viz 3: Annual Wage Percentiles (10th, 25th, Median, 75th, 90th) & Mean Across Roles', fontsize=13, fontweight='bold', pad=12)

legend_elements = [
    Patch(facecolor='#4c72b0', edgecolor='black', alpha=0.78, label='25th-75th Percentile (IQR)'),
    Line2D([0], [0], color='#555555', lw=1.8, label='10th-90th Percentile Range'),
    Line2D([0], [0], color='white', marker='|', markersize=10, markeredgecolor='black', lw=0, label='Median Wage (50th)'),
    Line2D([0], [0], marker='D', color='w', label='Mean Wage', markerfacecolor='#ff7f0e', markeredgecolor='black', markersize=8)
]
ax.legend(handles=legend_elements, loc='lower right', frameon=True, facecolor='white')
ax.set_xlim(40, 255)
plt.figtext(0.01, -0.03, 'Note: Capped values at $239.2K reflect BLS top-coded wage suppression (#). Source: BLS OEWS May 2023.', fontsize=9, style='italic')
plt.tight_layout()
plt.show()

## Visualisation 4: Bubble Scatter Plot – State-Level Data Scientists

- **Objective (relationship/exploration):** To explore how Data Scientist employment, mean wage and location quotient (LQ) relate across US states.

- **Why this visualisation:** Three quantitative variables need to be shown at once. Employment is on the x-axis (log scale, because state values range from under 100 to over 30,000), mean wage is on the y-axis, and LQ is encoded in bubble size and colour (Munzner, 2014). The red line marks the national mean wage ($119.0K) as a reference.

- **What I learned about AI employment:** Washington has the highest mean wage (≈$149K) despite modest employment (≈6K), while California has the largest workforce (>30K) with ≈$140K. New York (≈17K jobs, ≈$135K) and Virginia (≈$139K) also sit well above the national mean.

  Large employment does not guarantee high pay: Texas has about 21K Data Scientists but pays only ≈$109K, below the national mean, and Florida and Pennsylvania are also below it. The highest location quotient (above 3, the large yellow bubble at ≈3K jobs and ≈$129K) belongs to a mid-sized state rather than the biggest markets, so concentration of Data Scientists is not the same as total employment. Many states cluster at low employment and below-average pay.

In [5]:
df_states_ds = df_2023[
    (df_2023['AREA_TYPE'] == 2) &
    (df_2023['OCC_CODE'] == '15-2051')
].dropna(subset=['TOT_EMP', 'A_MEAN', 'LOC_QUOTIENT']).copy()

fig, ax = plt.subplots(figsize=(11.5, 6.5))
scatter = ax.scatter(
    df_states_ds['TOT_EMP'],
    df_states_ds['A_MEAN'] / 1000,
    s=df_states_ds['LOC_QUOTIENT'] * 95,
    c=df_states_ds['LOC_QUOTIENT'],
    cmap='plasma',
    alpha=0.82,
    edgecolors='black',
    linewidth=1.1
)

ax.set_xscale('log')
ax.set_xlabel('State Employment of Data Scientists (Log Scale)', fontsize=11, fontweight='bold')
ax.set_ylabel('Annual Mean Wage ($ USD Thousands)', fontsize=11, fontweight='bold')
ax.set_title('Viz 4: State-Level Data Scientist Employment, Wages & Location Quotient (May 2023)', fontsize=13, fontweight='bold', pad=12)

notable = df_states_ds.nlargest(7, 'TOT_EMP').index.union(df_states_ds.nlargest(5, 'A_MEAN').index)
for idx in notable:
    row = df_states_ds.loc[idx]
    ax.annotate(row['AREA_TITLE'], (row['TOT_EMP'], row['A_MEAN']/1000),
                xytext=(5, 5), textcoords='offset points', fontsize=8.5, fontweight='bold')

nat_mean = df_nat_2023[df_nat_2023['OCC_CODE'] == '15-2051']['A_MEAN'].values[0] / 1000
ax.axhline(nat_mean, color='#d62728', linestyle='--', linewidth=1.4, label=f'US National Mean (${nat_mean:,.1f}K)')

cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Location Quotient (Concentration vs. US Baseline = 1.0)', fontsize=10)
ax.legend(loc='upper left', frameon=True, facecolor='white')
plt.figtext(0.01, -0.02, 'Note: Bubble size & colour represent Location Quotient (LOC_QUOTIENT). Source: BLS OEWS May 2023.', fontsize=9, style='italic')
plt.tight_layout()
plt.show()